# RQ3 · Retractions in the biomedical literature
### Has the retraction rate risen over time, and which country and MeSH-area characteristics predict higher retraction *rates*?

**Design.**
- Unit of analysis: country × year.
- Sample: 2000-2023, countries with ≥ 50 publications/year in the full PubMed feed.
- Outcome: `retraction_rate = 100 × n_retracted / n_articles`.
- Regressors: `log(n_articles)` (scale  -  bigger producers get more retractions, mechanically), `year` trend within country FE (captures a country-specific slope), interaction with `field_is_biomed_core` (an indicator we build for medical vs adjacent-field records).
- Fixed effects: country.
- SEs: clustered at country.

**Why not simply count retractions?** Retraction *counts* are dominated
by publication scale. A country producing 200,000 papers/year will
retract more papers than one producing 500, but the *rate* comparison is
what tells us about integrity systems. We'll show both the level trend
and the rate regression.

**Data note.** In PubMed, retracted papers carry the publication type
`Retracted Publication`. The *retraction notice* is a separate record
of type `Retraction of Publication`. We count the retracted paper, not
the notice, so each retracted work is counted once, in its original
publication year.

**Compute footprint.** Two large PubMed pulls (all retractions ever, plus
a sample of the general literature for the denominator). The retraction
pull is small (~40K records); the denominator pull is stratified by year
so we don't pull the entire 35M-record corpus. OSG runs let us stratify
by MeSH-area for finer analyses.


In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ncbi_client import pubmed_query_to_dataframe, esearch
from src.regression import fixed_effects_panel


## 1. Pull retracted papers

In [ ]:
retractions = pubmed_query_to_dataframe(
    '"Retracted Publication"[pt]',
    mindate="2000/01/01", maxdate="2023/12/31",
    max_records=50000,
)
print(f"{len(retractions):,} retracted articles pulled")
retractions.head()


## 2. Pull annual PubMed counts (denominator, no XML)

We only need counts, not records  -  so a stripped-down ESearch per year is
enough. This is fast even without an API key.


In [ ]:
denom = []
for y in range(2000, 2024):
    n = esearch(f"{y}[dp]", retmax=0, use_history=False)["count"]
    denom.append({"year": y, "n_total_pubmed": n})
denom = pd.DataFrame(denom)
denom.plot(x="year", y="n_total_pubmed", marker="o", figsize=(9,3), title="Total PubMed publications per year")


## 3. Retraction volume vs rate over time

In [ ]:
year_ret = retractions.dropna(subset=["year"]).groupby("year").size().rename("n_retracted").reset_index()
year_ret = year_ret.merge(denom, on="year", how="left")
year_ret["retraction_rate_per_10k"] = 1e4 * year_ret["n_retracted"] / year_ret["n_total_pubmed"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
year_ret.plot(x="year", y="n_retracted", ax=ax1, marker="o", legend=False, title="Retracted papers per year")
year_ret.plot(x="year", y="retraction_rate_per_10k", ax=ax2, marker="o", color="crimson", legend=False,
              title="Retractions per 10,000 papers")
for ax in (ax1, ax2): ax.grid(alpha=0.3); ax.set_xlabel("")
fig.tight_layout()


## 4. Country × year panel of retraction rates

In [ ]:
# Country × year retracted counts
cy_ret = (
    retractions.dropna(subset=["year","country"])
    .groupby(["country","year"]).size().rename("n_retracted").reset_index()
)

# For the denominator by country we can't easily replicate ESearch per (country, year)
# without hitting rate limits; the honest approximation is to compute retraction share
# *within our retraction sample* and interpret it as such. For a rigorous denominator,
# switch this to the affiliation-tagged field-specific pull we used in RQ1.
country_totals = (
    retractions.dropna(subset=["country"]).groupby("country").size().rename("n_ret_total").reset_index()
)
cy_ret = cy_ret.merge(country_totals, on="country")
cy_ret["share_of_country_retractions"] = cy_ret["n_retracted"] / cy_ret["n_ret_total"]
cy_ret["log_n_retracted"] = np.log1p(cy_ret["n_retracted"])
cy_ret.head()


## 5. Fixed-effects regression on log retraction counts

In [ ]:
res = fixed_effects_panel(
    cy_ret,
    dependent="log_n_retracted",
    regressors=["year"],   # year enters as a linear trend WITHIN country
    entity="country",
    entity_effects=True, time_effects=False,   # if we included year FE it would absorb 'year'
    name="RQ3  -  Retraction count trend within country",
)
print(res.as_markdown())


## 6. Interpretation prompt

- The coefficient on `year` here is the *within-country* annual growth in log retraction counts. Exponentiate to get a growth rate: `exp(β) - 1`.
- Compare to the *global* trend in the plot from section 3. If the within-country coefficient is much smaller than the global slope, most of the global increase is being driven by cross-country composition shifts (more countries entering the sample).
- **Do not** interpret higher retraction counts as evidence of worse science without controlling for publication scale  -  countries publishing more papers will always retract more papers.


In [ ]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
year_ret.to_csv("../results/rq3_retractions_time_series.csv", index=False)
res.coef_table.to_csv("../results/rq3_retraction_country_trend_coefs.csv")
